# 1. Problema y datos

El objetivo es pronosticar la demanda real horaria de energía eléctrica del Sistema Interconectado Nacional de Colombia. Se formula una regresión temporal multivariada con salida directa de 24 intervalos y 168 horas de historia. El origen es diario a las 00:00 de Colombia. El estudio contempla la integración posterior de información documental.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'data/processed/panel_horario.parquet').exists())
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':.15})
panel = pq.read_table(ROOT/'data/processed/panel_horario.parquet', filters=[('inicio','<',pd.Timestamp('2022-01-01',tz='America/Bogota'))]).to_pandas()
assert panel.index.max() < pd.Timestamp('2022-01-01',tz='America/Bogota')
assert panel.index.is_monotonic_increasing and panel.index.is_unique
y = panel['y_demanda_real_kwh']
gwh = y / 1e6
registro = json.loads((ROOT/'data/processed/feature_registry.json').read_text(encoding='utf-8'))


El conjunto numérico abarca 2000–2025. El análisis exploratorio caracteriza desarrollo 2000–2021; las etiquetas de prueba quedan excluidas. La demanda se expresa en kWh por intervalo horario; las figuras utilizan GWh para facilitar su lectura. Las entradas observadas se alinean según disponibilidad y el calendario puede conocerse para el horizonte.

El [diccionario de datos](diccionario.md) define cada variable, sus unidades y disponibilidad. Incluye el panel horario, el corpus de titulares y los metadatos geográficos y poblacionales de las ocho ciudades.

In [2]:
inventario=pd.DataFrame([{'bloque':'Panel de desarrollo','filas':len(panel),'columnas':len(panel.columns),'inicio':str(panel.index.min()),'fin':str(panel.index.max())}]); display(inventario)
variables=pd.DataFrame.from_dict({k:v for k,v in registro.items() if not k.startswith('_')},orient='index'); display(variables.groupby('clase').size().rename('variables'))

,bloque,filas,columnas,inicio,fin
0,Panel de desarrollo,192864,58,2000-01-01 00:00:00-05:00,2021-12-31 23:00:00-05:00


clase
auditoria        1
known_future    23
past_only       33
target           1
Name: variables, dtype: int64

Las demandas comercial y real no se consideran intercambiables: la comercial queda excluida como entrada. Temperatura, humedad y precipitación proceden de reanálisis ERA5; sus observaciones pasadas complementan el panel, pero no representan pronósticos meteorológicos del horizonte. La hidrología es diaria. Los promedios urbanos ponderados utilizan población estática del censo 2018, documentada mediante una fuente secundaria.

**Decisión.** Conservar un registro explícito de rol y disponibilidad por variable.

**Impacto.** Permite distinguir entradas pasadas, calendario y objetivo.

**Riesgo.** Los rezagos supuestos y las revisiones retrospectivas limitan la disponibilidad histórica verificable.